In [ ]:
%matplotlib inline

# NOTE: %reset wipes every variable in the current kernel session, and normally
# prompts "Proceed (y/[n])?" interactively when run in a live notebook - kept
# here to ensure a clean run, but will hang if this notebook is ever run
# non-interactively (e.g. via jupyter nbconvert --execute).
%reset

import h5py
import multiprocessing
import numpy as np
import pandas as pd
import skfda
import sys

from joblib import Parallel, delayed
from pathlib import Path

projectRoot = Path().resolve().parent

sys.path.append(str(projectRoot / 'functions'))
import fNIRSData
import auxFuncChPt

import importlib
importlib.reload(fNIRSData)
importlib.reload(auxFuncChPt)


In [2]:
#User defined params
preProcPipeline = 'Standard' #'Standard, '015LPF, '025LPF'...

trialsIncluded = 'Full' # 'Hab', 'Nov, or 'Full'
firstTrial = 1
if trialsIncluded == 'Hab':
    lastTrial = 15
elif trialsIncluded == 'Nov':
    lastTrial = 20
else:
    lastTrial = 25

ttestSignificance = 0.05
cptSignificance = 0.01
# multiple comparison correction methods - e.g. 'b' = Bonferroni
ttestCorrectMethod = 'holm'
withinChanCorrectMethod = 'h' #'h' Holm
numBasis = 21
firstSamp = -40
lastSamp = 189
parcels = 'ALL' #'ROI' for ROI channels base on previous analysis; 'ALL' for all array channels
useLongRun = False #see autocorrelation test and ACF figures below!
lrCovLag = 0 #set to 0 to calculate h based on dataset size
useWildBinSeg = True

lineStyles = [ '-', '--', '-.', ':', 'solid', 'dashed', 'dashdot', 'dotted'] # for plotting

#for indexing later on
firstFunc = firstTrial-1
lastFunc = lastTrial-1


In [ ]:
parcelFilenameMetadata = projectRoot / 'dataRaw' / f'indigoSurfParcelMetadata{preProcPipeline}.csv'
parcelFilenameTimeseries = projectRoot / 'dataRaw' / f'indigoSurfParcelTrialsVlen{preProcPipeline}.h5'

outputFilenameChangepoint = projectRoot / 'data' / f'indigoIndividualSurfParcelChangepointDf{preProcPipeline}{trialsIncluded}.csv'
outputFilenameExcluded = projectRoot / 'data' / f'indigoIndividualParcelExcludedDf{preProcPipeline}{trialsIncluded}.csv'

In [ ]:
# Load metadata CSV
metadataDf = pd.read_csv(parcelFilenameMetadata)

# Infants whose HaND task was (partly) repeated across runs (overlapping trial numbers).
# KEEP_FIRST_RUN_ONLY = True  -> keep the first run only, so every trial is a first presentation.
# KEEP_FIRST_RUN_ONLY = False -> pool runs.
# Must match grpDotContrastsSurfParcel.ipynb and indDotTfceTTestSurfParcelSpatiotemporal4fcpt.ipynb.
# Split sessions (no overlapping trials) are always kept whole.
KEEP_FIRST_RUN_ONLY = True

runTrials = metadataDf.groupby(['Age', 'Infant', 'Run'])['Trial'].apply(set)
repeated = []
for (age, infant), g in runTrials.groupby(level=[0, 1]):
    if len(g) > 1 and set.intersection(*g.values):
        repeated.append((age, infant, g.index.get_level_values('Run').min()))
print(f"Repeated-run infants: {[(a, i) for a, i, _ in repeated]}")
if KEEP_FIRST_RUN_ONLY:
    for age, infant, firstRun in repeated:
        drop = (metadataDf['Age'] == age) & (metadataDf['Infant'] == infant) & (metadataDf['Run'] != firstRun)
        metadataDf = metadataDf[~drop]
    print('  -> first run kept for these infants')
else:
    print('  -> runs pooled for these infants')


In [ ]:

# load time series data
with h5py.File(parcelFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()

CHANGE POINT DETECTION

In [ ]:
# Setup
columnList = ['Infant',
              'Age',
              'Parcel',
              'Chromophore',
              'Recursion Level',
              'First Trial',
              'Last Trial',
              'Change Point',
              'Max. T_N',
              'p-value',
              'Corrected p-value']

# Initialise final storage
indChangepointRows = []
excludedCombinedValidLog = []
excludedCombinedTrialsLog = []
excludedChromValidLog = []
excludedChromTrialsLog = []

# Basis functions
basis = skfda.representation.basis.BSplineBasis(n_basis=numBasis)

# Load timeseries
with h5py.File(parcelFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()

# Get all ages
uniqueAges = metadataDf['Age'].unique()

# Loop over ages
for age in uniqueAges:  # Adjust slice as needed
    subsetDf = metadataDf[metadataDf['Age'] == age]
    uniqueInfants = subsetDf['Infant'].drop_duplicates().values
    uniqueParcels = subsetDf['Parcel'].drop_duplicates().values
    uniqueChroms = subsetDf['Chrom'].drop_duplicates().values

    for infant in uniqueInfants:
        print(f"\nInfant {infant} Parcel distribution:")
        print(subsetDf[subsetDf['Infant'] == infant]['Parcel'].value_counts())

    # Parallel loop over parcel-infant combinations
    results = Parallel(n_jobs=multiprocessing.cpu_count()-1)(delayed(fNIRSData.processIndividualRegionInfant)(
        age, infant, subsetDf, tsDataset, uniqueChroms, basis,
        firstTrial, lastTrial, firstSamp, lastSamp,
        numBasis, firstFunc, lastFunc,
        cptSignificance, lrCovLag, useLongRun, useWildBinSeg,
        fNIRSData, auxFuncChPt, regionType = 'Parcel', regionValue = parcel
    ) for parcel in uniqueParcels for infant in uniqueInfants)

    comboList = [(p, i) for p in uniqueParcels for i in uniqueInfants]

    # Merge outputs
    for (parcel, infant), (changepoints, log1, log2, log3, log4) in zip(comboList, results):
        print(f"Parcel {parcel}, Infant {infant} → {len(changepoints)} changepoints") # put in as a check as barely any participants had change points detected
        indChangepointRows.extend(changepoints)
        excludedCombinedValidLog.extend(log1)
        excludedCombinedTrialsLog.extend(log2)
        excludedChromValidLog.extend(log3)
        excludedChromTrialsLog.extend(log4)

# Save main output
dfOut = pd.DataFrame(indChangepointRows, columns=columnList)
dfOut.to_csv(outputFilenameChangepoint, index=False)

# Save exclusion logs
combinedExcludedDf = auxFuncChPt.saveExcludedLogs(
    excludedCombinedValidLog,
    excludedCombinedTrialsLog,
    excludedChromValidLog,
    excludedChromTrialsLog)
combinedExcludedDf.to_csv(outputFilenameExcluded, index=False)


In [ ]:
dfChangePoints = pd.read_csv(outputFilenameChangepoint)
dfChangePoints = dfChangePoints[dfChangePoints['Chromophore'] < 3]
dfChangePoints

In [ ]:
# Get the set of all distinct chromophore values in the whole dataframe
allChromophores = set(dfChangePoints['Chromophore'].unique())


# Function to check if the group contains all chromophores
def hasAllChromophores(group):
    return set(group).issuperset(allChromophores)

# Group by Age and Parcel, and check if each group contains all chromophores
result = (
    dfChangePoints.groupby(['Infant', 'Age', 'Parcel'])['Chromophore']
    .apply(hasAllChromophores)
    .reset_index(name='HasAllChromophores')
)

# Filter only the combinations where all chromophores are present
completeCombinations = result[result['HasAllChromophores']]

print(completeCombinations)